In [59]:
# Install required libraries
!pip install -q langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface langchain-chroma langchain-classic pypdf sentence-transformers


In [60]:
# Import libraries
import os

from google.colab import userdata
from langchain_core.prompts import MessagesPlaceholder
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

print("All libraries imported successfully!")

All libraries imported successfully!


In [61]:
# Create docs folder
os.makedirs("docs", exist_ok=True)

print(os.listdir("docs"))

['.ipynb_checkpoints', 'Travel Agency Marketing Strategy.pdf']


In [62]:
# Load PDF
loader = PyPDFLoader("docs/Travel Agency Marketing Strategy.pdf")

docs = loader.load()

print(f"Loaded {len(docs)} pages")

Loaded 11 pages


In [63]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
chunk_overlap=150
    )

chunks = splitter.split_documents(docs)

print(f"Loaded {len(chunks)} chunks")

Loaded 31 chunks


In [64]:
# Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [65]:
# Create vector database
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

In [66]:
# Get Groq API key
GROQ_API_KEY = userdata.get("areeba")

MODEL = "openai/gpt-oss-120b"

In [67]:
# Initialize Groq model
model = ChatGroq(
    model=MODEL,
    temperature=0.5,
    max_tokens=300,
    api_key=GROQ_API_KEY
)

In [68]:
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", """Rewrite the latest question as a standalone question
    using conversation history. Do not answer it."""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [69]:
history_aware_retriever = create_history_aware_retriever(
    model,
    retriever,
    contextualize_q_prompt
)

In [70]:
# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful Travel Agency Assistant.

    Answer using ONLY the provided PDF context.
    Give clear and detailed answers.
    Understand follow-up questions using conversation history.
    If information is not available in the context, reply exactly:
    I don't know

    Context:
    {context}"""),

    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [71]:
# Create document chain
doc_chain = create_stuff_documents_chain(
    model,
    prompt
)

# Create retrieval chain
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    doc_chain
)

In [73]:
response = conversational_rag.invoke(
    {"input": user_input},
    config=config
)

In [74]:
# Set up conversation memory
store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

In [75]:
# Create conversational RAG chain
conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [76]:
# Configure session
config = {
    "configurable": {
        "session_id": "user_123"
    }
}

In [77]:
# Chatbot
print("Bot is ready! Type 'exit' to stop. Type 'clear' to start fresh conversation.\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye!")
        break

    if user_input.lower() == "clear":
        store.pop("user_123", None)
        print("Bot: Memory cleared. Starting fresh!\n")
        continue

    response = conversational_rag.invoke(
        {"input": user_input},
        config=config
    )

    print("Bot:", response["answer"], "\n")

Bot is ready! Type 'exit' to stop. Type 'clear' to start fresh conversation.

You: tell me about travelling 
Bot: Travel serves many purposes, including holidays, business meetings, education, religious observances, family visits, and adventure experiences.  

With the rise of online booking platforms, travelers now have easier access to **discounts**, **hotel‑and‑transportation bundles**, and **flexible itineraries** that can be customized to fit different needs.  

**Young adults and students** especially tend to look for:
- **Budget‑friendly options**
- **Adventure activities**
- **Discounted rates**
- **Combined hotel and transport packages**
- **Flexible travel plans** that can adapt to changing schedules  

These features make travel more accessible and adaptable for a wide range of purposes and traveler groups. 

You: travelling market 
Bot: The travelling market is shaped by several key factors:

- **Intense competition** – Many agencies and online platforms vie for the same cu